# Tutorial 20 — The Engram Layer: Hash-Addressed n-gram Memory

**Series:** Training Language Models from Scratch: A Hacker's Guide  
**Part VII — DeepSeek Capstone**  
**Follows:** Tutorial 19 (Training NanoDeepSeek: GPT vs DeepSeek)  
**Precedes:** Tutorial 21 (Capstone: Ablating DeepSeek + Engram)

---

## The Memory Problem, Reformulated

In Tutorial 18 we distinguished four types of memory in language models:

| Memory type | Where it lives | How it's retrieved | Persistent? |
|---|---|---|---|
| **In-weights** | model parameters | implicit, always on | ✅ across steps |
| **In-context** | KV-cache / sequence | attention | ❌ context window only |
| **External** | retrieval DB | explicit lookup | ✅ |
| **n-gram / Engram** | hash table + embeddings | deterministic hash | ✅ across steps |

Engram occupies the fourth row. It is not attention and it is not a learned retrieval system — it is a *hash-addressed embedding table* keyed on $n$-gram surface forms. The key insight from the paper (DeepSeek, 2025):

> *This process essentially amounts to an expensive runtime reconstruction of a static lookup table, wasting valuable sequential depth on trivial operations that could otherwise be allocated to higher-level reasoning.*

The tradeoff: hash tables are O(1) to read but require a well-designed compression scheme to avoid collisions, and the table size must be chosen carefully relative to the vocabulary.

This tutorial builds **NanoEngram** from scratch — a simplified but functionally complete implementation. We then integrate it into NanoDeepSeek and measure its effect at inference time.

---

In [ ]:
import math
import unicodedata
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from dataclasses import dataclass, field
from typing import List, Dict, Tuple, Optional

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

---

## 1. Motivation: What Engram Solves

### 1.1 The Computation-vs-Retrieval Gap

Consider a fact: *"The capital of France is Paris."*

A standard transformer must recover this fact through a chain of attention + FFN operations — it has been *encoded* into weight matrices during training, but there is no direct read path. The fact "Paris" is reconstructed from distributed activations every forward pass.

An Engram layer provides a shortcut:
1. Hash the current $n$-gram context (e.g., "capital of France") to a table address.
2. Read the embedding at that address.
3. Gate it against the hidden state.
4. Add to residual stream *before* attention.

This is essentially *deterministic retrieval* — no learned router, no attention compute, O(1) lookup. The embedding at the address is learned during training to contain the contextually relevant signal for that surface form.

### 1.2 Where This Fits in the Architecture

In DeepSeek-V3-Engram (27B), the Engram layer is placed *before* attention at layers 2 and 15:

```
H ← H + Engram(H, input_ids)      # before Attention at layers 2 and 15
H ← H + Attention(H)
H ← H + MoE(H)
```

At our nano scale (6 layers), we place it at layers 2 and 6 (mirroring the relative positions).

---

## 2. CompressedTokenizer: Normalizing the Hash Space

Before hashing, input tokens are normalized so that *semantically equivalent* surface forms map to the same hash address. For example, `"Paris"`, `"paris"`, and `"PARIS"` should hit the same bucket.

Our nano tokenizer applies:
1. **NFKC normalization** — Unicode compatibility decomposition
2. **Lowercase** — case folding
3. **Whitespace collapse** — multiple spaces → single space

The original paper also applies NFD + StripAccents (removing accents from characters). We include that too for completeness, but it has minimal effect on English-only FineWeb-Edu.

The compressor builds a lookup table mapping original token IDs → compressed IDs, reducing the vocabulary by ~20% by collapsing case-equivalent and accent-equivalent types.

In [ ]:
from transformers import GPT2TokenizerFast

def _normalize(text: str) -> str:
    """NFKC → lowercase → NFD + strip accents → collapse whitespace."""
    # NFKC compatibility decomposition
    text = unicodedata.normalize('NFKC', text)
    text = text.lower()
    # NFD + strip combining characters (accents)
    text = unicodedata.normalize('NFD', text)
    text = ''.join(c for c in text if not unicodedata.combining(c))
    # Collapse whitespace
    text = ' '.join(text.split())
    return text


class CompressedTokenizer:
    """
    Builds a lookup table old_id → compressed_id by grouping tokens whose
    decoded text normalizes to the same string.

    This reduces the effective hash-space vocabulary without changing the
    model tokenizer — the original token IDs are still used for embeddings;
    only the Engram hashing step uses compressed IDs.
    """

    def __init__(self, tokenizer):
        self.tokenizer = tokenizer
        self.lookup_table, self.compressed_vocab_size = self._build()
        reduction = 1 - self.compressed_vocab_size / tokenizer.vocab_size
        print(f"CompressedTokenizer: {tokenizer.vocab_size} → "
              f"{self.compressed_vocab_size} tokens ({100*reduction:.1f}% reduction)")

    def _build(self) -> Tuple[np.ndarray, int]:
        V = self.tokenizer.vocab_size
        key2new: Dict[str, int] = {}
        lookup  = np.empty(V, dtype=np.int64)
        for tid in range(V):
            raw = self.tokenizer.decode([tid], skip_special_tokens=False)
            key = _normalize(raw) if '\ufffd' not in raw else raw
            if key not in key2new:
                key2new[key] = len(key2new)
            lookup[tid] = key2new[key]
        return lookup, len(key2new)

    def __call__(self, input_ids: np.ndarray) -> np.ndarray:
        """Map a (B, T) int64 array of token IDs to compressed IDs."""
        ids    = np.asarray(input_ids, dtype=np.int64)
        flat   = ids.reshape(-1)
        mapped = self.lookup_table[flat]
        return mapped.reshape(ids.shape)


# Build once and share
base_tokenizer = GPT2TokenizerFast.from_pretrained('gpt2')
base_tokenizer.pad_token = base_tokenizer.eos_token
comp_tokenizer  = CompressedTokenizer(base_tokenizer)

---

## 3. NgramHashMapping: Deterministic XOR Hashing

Given a sequence of compressed token IDs $[x_0, x_1, \ldots, x_{T-1}]$, the $n$-gram hash for position $t$ is:

$$h_t^{(n)} = \bigoplus_{k=0}^{n-1} (x_{t-k} \cdot m_k)$$

where:
- $m_k$ are seeded odd-integer multipliers (different per layer to avoid inter-layer collisions)
- $\oplus$ is XOR
- $x_{t-k}$ uses left-padding with the pad token for positions before the sequence start
- The result is taken modulo a per-head prime to reduce collisions[^xorprime]

For NanoEngram we compute $n=2$ and $n=3$ grams (bigrams and trigrams), each with `n_head` independent hash functions (using different primes for modular reduction).

[[[The prime moduli are chosen to be just above the per-ngram vocabulary size]{.underline}, with all primes across all layers and heads kept distinct — this is the same strategy as the original paper.]{.mark}

[^xorprime]: A prime modulus minimises systematic bias in XOR hash distributions. Birthday-bound collision probability for a table of size $N$ and $n$-gram space of size $V^n$ is approximately $V^n / (2N)$. For bigrams over GPT-2's vocabulary ($V \approx 50{,}000$, $N = 50{,}000$), this is $\approx 25{,}000{,}000/100{,}000 = 250$ expected collisions per 1{,}000 unique bigrams — probabilistic but bounded, and mitigated by the multi-head design averaging over independent hash functions.

In [ ]:
def _next_prime(n: int, seen: set) -> int:
    """Find the next prime > n not already used."""
    from sympy import isprime
    c = n + 1
    while not isprime(c) or c in seen:
        c += 1
    return c


@dataclass
class NanoEngramConfig:
    # Hash table size per ngram type (bigrams, trigrams)
    engram_vocab_size: List[int] = field(default_factory=lambda: [50_000, 50_000])
    max_ngram_size:    int   = 3          # n = 2 and n = 3
    n_embed:           int   = 64         # embedding dimension per head per ngram
    n_head:            int   = 4          # hash heads per ngram order
    layer_ids:         List[int] = field(default_factory=lambda: [2, 6])
    pad_id:            int   = 0
    seed:              int   = 42


class NgramHashMapping:
    """
    For each Engram layer and each position (B, T), returns a (B, T, n_heads_total)
    integer tensor of hash indices — one per hash head per n-gram order.

    n_heads_total = (max_ngram_size - 1) * n_head
                  =  2 (orders: bigram, trigram) × n_head
    """

    def __init__(self, cfg: NanoEngramConfig, comp_tokenizer: CompressedTokenizer):
        self.cfg  = cfg
        self.comp = comp_tokenizer

        # Build seeded multipliers per layer (odd integers to prevent zero collapse)
        self.layer_multipliers: Dict[int, np.ndarray] = {}
        MAX_LONG   = np.iinfo(np.int64).max
        HALF_BOUND = MAX_LONG // 2
        for lid in cfg.layer_ids:
            rng  = np.random.default_rng(int(cfg.seed + 10007 * lid))
            raw  = rng.integers(0, HALF_BOUND, size=(cfg.max_ngram_size,), dtype=np.int64)
            self.layer_multipliers[lid] = raw * 2 + 1   # make odd

        # Build per-layer, per-ngram, per-head prime moduli (all globally distinct)
        seen_primes: set = set()
        self.primes: Dict[int, List[List[int]]] = {}   # [layer][n_order][head]
        for lid in cfg.layer_ids:
            layer_primes = []
            for ni in range(cfg.max_ngram_size - 1):   # bigram, trigram
                head_primes = []
                start = cfg.engram_vocab_size[ni] - 1
                for _ in range(cfg.n_head):
                    p = _next_prime(start, seen_primes)
                    seen_primes.add(p)
                    head_primes.append(p)
                    start = p
                layer_primes.append(head_primes)
            self.primes[lid] = layer_primes

    def hash(self, input_ids: np.ndarray, layer_id: int) -> np.ndarray:
        """
        Returns: (B, T, n_heads_total) int64
        where n_heads_total = (max_ngram_size-1) * n_head
        """
        x    = self.comp(input_ids)                   # (B, T) compressed
        B, T = x.shape
        muls = self.layer_multipliers[layer_id]       # (max_ngram_size,)

        # Shifted copies: shift_k[:, k:T] = x[:, :T-k], shift_k[:, :k] = pad
        def shift_k(k: int):
            if k == 0: return x
            padded = np.pad(x, ((0,0),(k,0)), mode='constant',
                            constant_values=self.cfg.pad_id)
            return padded[:, :T]

        shifts = [shift_k(k) for k in range(self.cfg.max_ngram_size)]

        all_hashes = []
        for ni in range(self.cfg.max_ngram_size - 1):   # 0→bigram, 1→trigram
            n = ni + 2                                   # actual n in n-gram
            mix = shifts[0] * muls[0]
            for k in range(1, n):
                mix = np.bitwise_xor(mix, shifts[k] * muls[k])
            for hj, prime in enumerate(self.primes[layer_id][ni]):
                all_hashes.append((mix % prime).astype(np.int64))

        return np.stack(all_hashes, axis=2)   # (B, T, n_heads_total)


# Smoke test
nano_engram_cfg = NanoEngramConfig()
hasher = NgramHashMapping(nano_engram_cfg, comp_tokenizer)
test_ids = np.array([[1, 25, 1337, 50, 8, 300]], dtype=np.int64)
h = hasher.hash(test_ids, layer_id=2)
print(f"Hash output shape: {h.shape}   expected: (1, 6, {(nano_engram_cfg.max_ngram_size-1)*nano_engram_cfg.n_head})")

---

## 4. MultiHeadEmbedding: Shared Table with Per-Head Offsets

Rather than $H$ separate embedding tables, we use one large table of size $\sum_i N_i$ and address head $i$ by adding its offset:

$$\text{offset}_i = \sum_{j < i} N_j$$

This is equivalent to separate tables but more memory-efficient (single `nn.Embedding` call; PyTorch fuses the lookup).

In [ ]:
class MultiHeadEmbedding(nn.Module):
    """
    Single embedding table of size sum(N_i) with per-head offsets.

    Input:  (B, T, n_heads_total) int64
    Output: (B, T, n_heads_total, d_embed)
    """

    def __init__(self, vocab_sizes: List[int], d_embed: int):
        super().__init__()
        self.n_heads  = len(vocab_sizes)
        self.d_embed  = d_embed
        # Cumulative offsets registered as a non-trainable buffer
        offsets = [0]
        for v in vocab_sizes[:-1]:
            offsets.append(offsets[-1] + v)
        self.register_buffer('offsets', torch.tensor(offsets, dtype=torch.long))
        self.embedding = nn.Embedding(sum(vocab_sizes), d_embed)

    def forward(self, idx: torch.Tensor) -> torch.Tensor:
        """idx: (B, T, n_heads) → (B, T, n_heads, d_embed)"""
        # offsets: (n_heads,) → broadcast over (B, T, n_heads)
        shifted = idx + self.offsets  # (B, T, n_heads)
        return self.embedding(shifted)  # (B, T, n_heads, d_embed)


# Verify shapes
vocab_sizes = [
    p
    for ni in range(nano_engram_cfg.max_ngram_size - 1)
    for p in hasher.primes[2][ni]
]
mhe = MultiHeadEmbedding(vocab_sizes, d_embed=nano_engram_cfg.n_embed)
test_idx = torch.from_numpy(h)   # (1, 6, 8)
emb_out  = mhe(test_idx)
print(f"MultiHeadEmbedding output shape: {tuple(emb_out.shape)}")
print(f"Total embedding parameters: {sum(p.numel() for p in mhe.parameters())/1e3:.1f}K")

---

## 5. Gating: Routing the Hash Embedding into the Residual Stream

[[[The Engram output is not added unconditionally.]{.mark}]{.underline} A gate decides how much of the hash embedding is relevant given the current hidden state $h_t$.

The gate is a dot-product of the (RMSNorm-normalized) query $h_t$ and key $k_t$ (projected from the hash embedding), followed by an unusual *sqrt-sigmoid* activation that makes the gate more conservative near 0 and sharper near ±1:

$$\alpha_t = \sigma\!\left(\sqrt{|s_t|} \cdot \text{sign}(s_t)\right), \qquad s_t = \frac{(\hat{h}_t \cdot \hat{k}_t)}{\sqrt{d}}$$

where $\hat{\cdot}$ denotes RMSNorm. The final gated output is:

$$\text{out}_t = \alpha_t \cdot W_V \, e_t$$

where $e_t$ is the flattened hash embedding for position $t$.

In the paper's full implementation, there are $M$ independent key projections (one per hyper-connection head). That 4D structure is described in the **mHC sidebar** below; in NanoEngram we collapse the hc_mult dimension.

---

> **Sidebar: Multi-Head Hyper-Connection (mHC) — What We're Skipping**
>
> In the production DeepSeek-Engram implementation (27B scale), the residual stream is *not* a 3D tensor `(B, L, D)`. It is a 4D tensor `(B, L, M, D)` where `M = hc_mult = 4` is the number of *hyper-connection* heads. Each hyper-connection head maintains an independent copy of the residual that is mixed at each block via learned gates — think of it as running `M` parallel residual streams that exchange information.
>
> In the Engram layer, mHC means:
> - There are `M` independent key projections $W_K^{(m)}$ and `M` corresponding gates $\alpha_t^{(m)}$.
> - The gated value is broadcast across all `M` heads: `value: (B, L, M, D)`.
> - `ShortConv` operates over the 4D `(B, L, M, D)` tensor with grouped convolutions.
>
> **Why we skip it in NanoEngram:**
> 1. The orthogonality constraint between hyper-connection heads requires Riemannian SGD or a custom optimizer — standard AdamW doesn't enforce it.
> 2. At nano scale (6 layers, d=384), maintaining `M=4` residual streams would roughly quadruple residual memory and likely collapse to a degenerate solution within our token budget.
> 3. The 4D tensor requires non-trivial shape bookkeeping throughout every `DSBlock`, making the code significantly harder to follow.
>
> **What we do instead:** We collapse `M=1` and expand the hidden state to `4*d_model` before the key projection — `W_K: (d, 4d)`, same FLOPs as `M` independent projections, but all in 3D. [[[This is a valid single-head approximation that captures the routing logic while keeping all tensors 3D.]{.mark}]{.underline}
>
> **What this costs:** The orthogonality inductive bias is lost. The `M` heads in the real implementation are encouraged to be diverse by the Riemannian constraint, providing complementary routing signals. Our single expanded projection may not learn this diversity without the structural constraint.

---

---

## 6. ShortConv: Causal Local Aggregation

After gating, a short depthwise causal convolution smoothes the gated values across a local window:

$$Y = \text{SiLU}(\text{CausalConv1D}_{\text{dw}}(\text{RMSNorm}(V))) + V$$

- **Kernel size 4, dilation = max_ngram_size = 3**: receptive field = $(4-1) \times 3 + 1 = 10$ positions
- **Depthwise**: groups = channels (no cross-channel mix in conv, only via $W_V$ above)
- **Causal**: we pad only on the left and trim the right to length $T$

This allows the model to aggregate information from recent n-gram hits before injecting into the residual stream.

In [ ]:
class ShortConv(nn.Module):
    """
    Causal depthwise Conv1D + SiLU, operating on (B, T, d_model).

    In the full Engram implementation this operates over (B, T, M, D) with
    grouped norms per M head. Here we collapse M=1 and work in 3D.
    """

    def __init__(self, d: int, kernel_size: int = 4, dilation: int = 3):
        super().__init__()
        self.norm = nn.RMSNorm(d)
        padding   = (kernel_size - 1) * dilation
        self.conv = nn.Conv1d(
            in_channels=d,
            out_channels=d,
            kernel_size=kernel_size,
            groups=d,
            dilation=dilation,
            padding=padding,
            bias=False,
        )
        self.act  = nn.SiLU()

    def forward(self, v: torch.Tensor) -> torch.Tensor:
        """v: (B, T, d)"""
        B, T, d  = v.shape
        normed   = self.norm(v)                     # (B, T, d)
        x_bct    = normed.transpose(1, 2)           # (B, d, T)
        y_bct    = self.conv(x_bct)[..., :T]        # trim right padding → (B, d, T)
        return self.act(y_bct.transpose(1, 2)) + v  # (B, T, d)

---

## 7. NanoEngram: Putting It Together

In [ ]:
class NanoEngram(nn.Module):
    """
    Single Engram layer for one specific Transformer layer position.

    Flow:
      e  = MultiHeadEmbedding(hash(input_ids))  # (B,T,n_heads,n_embed)
      ef = e.flatten(-2)                         # (B,T, n_heads*n_embed)
      V  = W_V(ef)                               # (B,T, d_model)
      k  = W_K(ef)                               # (B,T, 4*d_model)  [mHC collapsed]
      q  = h                                     # (B,T, d_model)
      s  = RMSNorm(q) · RMSNorm(k) / sqrt(d)
      α  = sigmoid(sqrt(|s|) * sign(s))          # sqrt-sigmoid gate
      out = ShortConv(α * V) + α * V
    """

    def __init__(self, layer_id: int, cfg: NanoEngramConfig, d_model: int,
                 hasher: NgramHashMapping, hc_mult: int = 4):
        super().__init__()
        self.layer_id = layer_id
        self.hasher   = hasher
        self.hc_mult  = hc_mult
        self.d        = d_model

        # vocab sizes for this layer's hash heads (bigram heads + trigram heads)
        vocab_sizes = [
            p
            for ni in range(cfg.max_ngram_size - 1)
            for p in hasher.primes[layer_id][ni]
        ]
        self.n_heads  = len(vocab_sizes)          # = (max_ngram_size-1) * n_head
        self.n_embed  = cfg.n_embed
        engram_dim    = self.n_heads * cfg.n_embed

        self.mhe      = MultiHeadEmbedding(vocab_sizes, cfg.n_embed)
        self.W_V      = nn.Linear(engram_dim, d_model,          bias=False)
        self.W_K      = nn.Linear(engram_dim, hc_mult * d_model, bias=False)  # mHC collapsed
        self.norm_k   = nn.RMSNorm(hc_mult * d_model)
        self.norm_q   = nn.RMSNorm(d_model)
        self.short_conv = ShortConv(d_model, kernel_size=cfg.kernel_size if hasattr(cfg,'kernel_size') else 4,
                                    dilation=cfg.max_ngram_size)

    def forward(self, h: torch.Tensor, input_ids: np.ndarray) -> torch.Tensor:
        """
        h:         (B, T, d_model) hidden state
        input_ids: (B, T) numpy int64, ORIGINAL token IDs (not compressed)
        Returns:   (B, T, d_model) Engram contribution (to be added to h)
        """
        B, T, d = h.shape
        dev      = h.device

        # 1. Hash → embeddings ────────────────────────────────────────────────
        ids_np   = input_ids.cpu().numpy() if isinstance(input_ids, torch.Tensor) else input_ids
        hash_idx = self.hasher.hash(ids_np, self.layer_id)              # (B,T,n_heads) np.int64
        hash_t   = torch.from_numpy(hash_idx).to(dev)                   # (B,T,n_heads)
        emb      = self.mhe(hash_t)                                      # (B,T,n_heads,n_embed)
        ef       = emb.flatten(start_dim=-2)                             # (B,T, n_heads*n_embed)

        # 2. Project to value and key ─────────────────────────────────────────
        V  = self.W_V(ef)                                                # (B,T, d)
        K  = self.W_K(ef)                                                # (B,T, hc_mult*d)

        # 3. sqrt-sigmoid gate ────────────────────────────────────────────────
        # Collapse hc_mult: sum-pool the expanded key to d for dot-product
        K_pool = K.view(B, T, self.hc_mult, d).sum(dim=2)               # (B,T,d)
        nq = self.norm_q(h)                                              # (B,T,d)
        nk = self.norm_k(K).view(B,T,self.hc_mult,d).mean(dim=2)        # (B,T,d)
        s  = (nq * nk).sum(dim=-1, keepdim=True) / math.sqrt(d)         # (B,T,1)
        a  = s.abs().clamp(1e-9).sqrt() * s.sign()                      # sqrt-sigmoid input
        gate = a.sigmoid()                                               # (B,T,1)

        # 4. Gated value + ShortConv ──────────────────────────────────────────
        gated = gate * V                                                  # (B,T,d)
        out   = self.short_conv(gated)                                   # (B,T,d)
        return out


# Verify shapes
nano_engram_cfg = NanoEngramConfig()
hasher2 = NgramHashMapping(nano_engram_cfg, comp_tokenizer)
engram_layer2 = NanoEngram(layer_id=2, cfg=nano_engram_cfg, d_model=384,
                            hasher=hasher2)

dummy_h   = torch.randn(2, 16, 384)
dummy_ids = np.random.randint(0, 50257, size=(2, 16), dtype=np.int64)
out = engram_layer2(dummy_h, dummy_ids)
print(f"NanoEngram output shape: {tuple(out.shape)}")
print(f"Engram layer params: {sum(p.numel() for p in engram_layer2.parameters())/1e3:.1f}K")

---

## 8. Integrating NanoEngram into NanoDeepSeek

We assemble `NanoDeepSeekWithEngram` — identical to NanoDeepSeek from Tutorial 19 except that blocks at `layer_ids` have an Engram module prepended. The residual integration is:

```
if layer_id in engram_layer_ids:
    x = x + NanoEngram(x, input_ids)
x = x + NanoMLA(RMSNorm(x))
x = x + NanoMoE(RMSNorm(x))
```

In [ ]:
# ── Paste NanoDeepSeek components from Tutorial 18/19 ─────────────────────────

class RMSNorm(nn.Module):
    def __init__(self, d: int, eps: float = 1e-6):
        super().__init__()
        self.eps   = eps
        self.gamma = nn.Parameter(torch.ones(d))
    def forward(self, x):
        return x / x.pow(2).mean(-1, keepdim=True).add(self.eps).sqrt() * self.gamma

def make_rope_cache(max_seq_len, d_head, device):
    theta = 1.0 / (10000 ** (torch.arange(0, d_head, 2, device=device).float() / d_head))
    pos   = torch.arange(max_seq_len, device=device).float()
    freqs = torch.cat([torch.outer(pos, theta)] * 2, dim=-1)
    return freqs.cos()[None, None], freqs.sin()[None, None]

def apply_rope(x, cos, sin):
    x1, x2 = x[..., ::2], x[..., 1::2]
    return x * cos + torch.stack([-x2, x1], dim=-1).flatten(-2) * sin

def causal_mask(T, device):
    return torch.triu(torch.ones(T, T, dtype=torch.bool, device=device), 1)[None, None]


@dataclass
class NanoDeepSeekConfig:
    vocab_size:     int   = 50257
    d_model:        int   = 384
    n_layers:       int   = 6
    n_heads:        int   = 6
    d_compressed:   int   = 96
    d_ffn:          int   = 1024
    n_shared:       int   = 1
    n_routed:       int   = 8
    top_k:          int   = 2
    max_seq_len:    int   = 256
    aux_loss_coeff: float = 1e-2

class SwiGLU(nn.Module):
    def __init__(self, d, d_ff):
        super().__init__()
        self.W1=nn.Linear(d,d_ff,bias=False); self.W3=nn.Linear(d,d_ff,bias=False)
        self.W2=nn.Linear(d_ff,d,bias=False)
    def forward(self, x): return self.W2(F.silu(self.W1(x))*self.W3(x))

class NanoMLA(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        d,nh,dc=cfg.d_model,cfg.n_heads,cfg.d_compressed
        self.nh=nh;self.dh=d//nh;self.d=d
        self.W_c=nn.Linear(d,dc,bias=False);self.W_K=nn.Linear(dc,d,bias=False)
        self.W_V=nn.Linear(dc,d,bias=False);self.W_Q=nn.Linear(d,d,bias=False)
        self.W_O=nn.Linear(d,d,bias=False)
        cos,sin=make_rope_cache(cfg.max_seq_len,self.dh,'cpu')
        self.register_buffer('cos',cos);self.register_buffer('sin',sin)
    def forward(self,x,mask=None):
        B,T,_=x.shape
        c_kv=self.W_c(x);K,V=self.W_K(c_kv),self.W_V(c_kv);Q=self.W_Q(x)
        def mh(t): return t.view(B,T,self.nh,self.dh).transpose(1,2)
        Q,K,V=mh(Q),mh(K),mh(V)
        cos,sin=self.cos[:,:,:T,:].to(x.device),self.sin[:,:,:T,:].to(x.device)
        Q,K=apply_rope(Q,cos,sin),apply_rope(K,cos,sin)
        sc=Q@K.transpose(-2,-1)/math.sqrt(self.dh)
        if mask is not None: sc=sc.masked_fill(mask,float('-inf'))
        out=(F.softmax(sc,dim=-1)@V).transpose(1,2).reshape(B,T,self.d)
        return self.W_O(out)

class NanoMoE(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.n_shared=cfg.n_shared;self.n_routed=cfg.n_routed
        self.top_k=cfg.top_k;self.alpha=cfg.aux_loss_coeff
        self.shared=nn.ModuleList([SwiGLU(cfg.d_model,cfg.d_ffn) for _ in range(cfg.n_shared)])
        self.experts=nn.ModuleList([SwiGLU(cfg.d_model,cfg.d_ffn) for _ in range(cfg.n_routed)])
        self.router=nn.Linear(cfg.d_model,cfg.n_routed,bias=False)
    def forward(self,x):
        B,T,d=x.shape;xf=x.view(B*T,d)
        out=sum(e(xf) for e in self.shared)
        probs=F.softmax(self.router(xf),dim=-1)
        topk_vals,topk_idx=torch.topk(probs,self.top_k,dim=-1)
        gates=topk_vals/(topk_vals.sum(-1,keepdim=True)+1e-9)
        routed=torch.zeros_like(xf)
        for ki in range(self.top_k):
            for ei in range(self.n_routed):
                m=topk_idx[:,ki]==ei
                if m.any(): routed[m]+=gates[:,ki:ki+1][m]*self.experts[ei](xf[m])
        with torch.no_grad():
            f=F.one_hot(topk_idx[:,0],self.n_routed).float().mean(0)
        aux=self.alpha*self.n_routed*(f*probs.mean(0)).sum()
        return (out+routed).view(B,T,d),aux,probs.detach()


# ── DSBlock with optional Engram ───────────────────────────────────────────────

class DSBlockWithEngram(nn.Module):
    def __init__(self, layer_id: int, ds_cfg: NanoDeepSeekConfig,
                 engram: Optional[NanoEngram] = None):
        super().__init__()
        self.layer_id = layer_id
        self.engram   = engram     # None for layers without Engram
        self.n1       = RMSNorm(ds_cfg.d_model)
        self.attn     = NanoMLA(ds_cfg)
        self.n2       = RMSNorm(ds_cfg.d_model)
        self.moe      = NanoMoE(ds_cfg)

    def forward(self, x: torch.Tensor, input_ids: Optional[np.ndarray], mask=None):
        if self.engram is not None and input_ids is not None:
            x = x + self.engram(x, input_ids)
        x = x + self.attn(self.n1(x), mask)
        moe_out, aux, probs = self.moe(self.n2(x))
        x = x + moe_out
        return x, aux, probs


class NanoDeepSeekWithEngram(nn.Module):
    def __init__(self, ds_cfg: NanoDeepSeekConfig, engram_cfg: NanoEngramConfig,
                 comp_tok: CompressedTokenizer):
        super().__init__()
        self.ds_cfg = ds_cfg

        # Build shared hasher (covers all engram layer_ids)
        self.hasher = NgramHashMapping(engram_cfg, comp_tok)

        self.emb   = nn.Embedding(ds_cfg.vocab_size, ds_cfg.d_model)
        self.blocks = nn.ModuleList()
        for lid in range(ds_cfg.n_layers):
            engram = None
            if lid in engram_cfg.layer_ids:
                engram = NanoEngram(
                    layer_id=lid,
                    cfg=engram_cfg,
                    d_model=ds_cfg.d_model,
                    hasher=self.hasher,
                )
            self.blocks.append(DSBlockWithEngram(lid, ds_cfg, engram))
        self.norm  = RMSNorm(ds_cfg.d_model)
        self.head  = nn.Linear(ds_cfg.d_model, ds_cfg.vocab_size, bias=False)
        self.head.weight = self.emb.weight

    def forward(self, idx: torch.Tensor, targets=None):
        B, T = idx.shape
        x    = self.emb(idx)
        mask = causal_mask(T, idx.device)
        ids_np    = idx.cpu().numpy()
        total_aux = torch.tensor(0., device=idx.device)
        all_probs = []
        for blk in self.blocks:
            x, aux, probs = blk(x, ids_np, mask)
            total_aux = total_aux + aux
            all_probs.append(probs)
        logits = self.head(self.norm(x))
        loss   = None
        if targets is not None:
            lm   = F.cross_entropy(logits.view(-1, logits.size(-1)), targets.view(-1))
            loss = lm + total_aux
        return logits, loss, all_probs


# Build and verify
ds_cfg  = NanoDeepSeekConfig()
eg_cfg  = NanoEngramConfig()
model_eg = NanoDeepSeekWithEngram(ds_cfg, eg_cfg, comp_tokenizer).to(device)

total_p   = sum(p.numel() for p in model_eg.parameters())
engram_p  = sum(p.numel() for blk in model_eg.blocks if blk.engram
                for p in blk.engram.parameters())
print(f"NanoDeepSeekWithEngram — total params: {total_p/1e6:.2f}M  "
      f"(Engram: {engram_p/1e3:.1f}K)")

# Forward pass
dummy = torch.randint(0, 50257, (2, 32)).to(device)
tgt   = torch.randint(0, 50257, (2, 32)).to(device)
logits, loss, probs = model_eg(dummy, tgt)
print(f"Forward pass OK — logits: {tuple(logits.shape)}, loss: {loss.item():.4f}")

---

## 9. Gating Heatmap: What Does Engram Attend To?

We visualize the gate values $\alpha_t$ for each position in a sentence, averaging over heads. [[[High gate values indicate that the hash embedding for that $n$-gram context carried a strong signal]{.underline} — typically proper nouns, technical terms, or rare multi-word expressions.]{.mark}

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors

@torch.no_grad()
def get_gate_values(model: NanoDeepSeekWithEngram, text: str) -> Dict[int, np.ndarray]:
    """Return gate values (T,) for each Engram layer given a text prompt."""
    tokens = base_tokenizer.encode(text)
    idx    = torch.tensor(tokens, dtype=torch.long, device=device).unsqueeze(0)
    B, T   = idx.shape
    x      = model.emb(idx)
    ids_np = idx.cpu().numpy()
    mask   = causal_mask(T, device)

    gates_by_layer: Dict[int, np.ndarray] = {}

    for blk in model.blocks:
        if blk.engram is not None:
            eg   = blk.engram
            lid  = blk.layer_id
            hash_idx = eg.hasher.hash(ids_np, lid)
            hash_t   = torch.from_numpy(hash_idx).to(device)
            emb      = eg.mhe(hash_t).flatten(start_dim=-2)
            K        = eg.W_K(emb)
            nk       = eg.norm_k(K).view(B,T,eg.hc_mult,eg.d).mean(dim=2)
            nq       = eg.norm_q(x)
            s        = (nq * nk).sum(-1, keepdim=True) / math.sqrt(eg.d)
            a        = s.abs().clamp(1e-9).sqrt() * s.sign()
            gate     = a.sigmoid().squeeze(-1).squeeze(0)   # (T,)
            gates_by_layer[lid] = gate.cpu().numpy()

        # run block for hidden state update
        x, _, _ = blk(x, ids_np, mask)

    return gates_by_layer


sentence = "Only Alexander the Great could tame the horse Bucephalus."
tokens   = base_tokenizer.encode(sentence)
token_strs = [base_tokenizer.decode([t]) for t in tokens]

gate_vals = get_gate_values(model_eg, sentence)

fig, axes = plt.subplots(len(gate_vals), 1, figsize=(14, 2.5 * len(gate_vals)))
if len(gate_vals) == 1:
    axes = [axes]

for ax, (lid, gates) in zip(axes, sorted(gate_vals.items())):
    im = ax.imshow(gates[None, :], aspect='auto', cmap='YlOrRd',
                   vmin=0, vmax=1)
    ax.set_xticks(range(len(token_strs)))
    ax.set_xticklabels(token_strs, rotation=45, ha='right', fontsize=9)
    ax.set_yticks([])
    ax.set_title(f'Engram gate α — Layer {lid}')
    plt.colorbar(im, ax=ax)

plt.suptitle(f'"{sentence}"', fontsize=10)
plt.tight_layout()
plt.savefig('t20_engram_gates.png', dpi=150)
plt.show()
print("Saved t20_engram_gates.png")

> **What to look for:** Proper nouns (*Alexander*, *Bucephalus*), rare multi-word spans (*Great could tame*, *tame the horse*), and the end-of-sentence context will often show higher gate values after training. At initialization (as shown here), gates will be near 0.5 uniformly since the embeddings are random — this visualization becomes meaningful after training.

---

## 10. Sensitivity Analysis: Ablating Engram at Inference

We zero-out the Engram contribution — setting all gates to 0 — and measure the perplexity increase on a held-out sample. This quantifies how much the model relies on Engram vs pure attention.

In [ ]:
from datasets import load_dataset

@torch.no_grad()
def eval_perplexity(model, n_batches: int = 20, seq_len: int = 64,
                    batch_size: int = 8, ablate_engram: bool = False):
    """Estimate perplexity on a few FineWeb-Edu batches."""
    if ablate_engram:
        # Monkey-patch: replace all NanoEngram.forward with zero-return
        original_forwards = {}
        for blk in model.blocks:
            if blk.engram is not None:
                eg = blk.engram
                original_forwards[id(eg)] = eg.forward
                # Return zeros of the right shape
                def zero_forward(h, input_ids, _eg=eg):
                    return torch.zeros_like(h)
                eg.forward = zero_forward

    model.eval()
    total_loss, n_batches_done = 0.0, 0

    try:
        from torch.utils.data import DataLoader
        ds = load_dataset('HuggingFaceFW/fineweb-edu', name='sample-10BT',
                           split='train', streaming=True, trust_remote_code=True)
        buf = []
        for doc in ds:
            ids = base_tokenizer.encode(doc['text'], add_special_tokens=False)
            buf.extend(ids)
            while len(buf) >= seq_len + 1 and n_batches_done < n_batches:
                chunk = buf[:seq_len+1]; buf = buf[seq_len+1:]
                inp = torch.tensor(chunk[:-1], dtype=torch.long, device=device).unsqueeze(0)
                tgt = torch.tensor(chunk[1:],  dtype=torch.long, device=device).unsqueeze(0)
                _, loss, _ = model(inp, tgt)
                total_loss += loss.item()
                n_batches_done += 1
            if n_batches_done >= n_batches:
                break
    finally:
        if ablate_engram:
            for blk in model.blocks:
                if blk.engram is not None:
                    eg = blk.engram
                    if id(eg) in original_forwards:
                        eg.forward = original_forwards[id(eg)]

    avg_loss = total_loss / max(n_batches_done, 1)
    return math.exp(avg_loss)


print("Evaluating (note: model is untrained — values are random-init baseline)")
ppl_with    = eval_perplexity(model_eg, ablate_engram=False)
ppl_without = eval_perplexity(model_eg, ablate_engram=True)

print(f"Perplexity WITH    Engram: {ppl_with:.2f}")
print(f"Perplexity WITHOUT Engram: {ppl_without:.2f}")
print(f"ΔPP (Engram contribution): {ppl_without - ppl_with:+.2f} ({100*(ppl_without/ppl_with-1):+.1f}%)")
print("\n(For a trained model, expect 2–8% perplexity increase when ablating.")
print("Tutorial 21 measures this systematically across training checkpoints.)")

---

## Summary

| Component | Purpose | Key design |
|---|---|---|
| `CompressedTokenizer` | Normalize hash space | NFKC + lowercase + accent strip |
| `NgramHashMapping` | Map $n$-grams → table addresses | Seeded XOR + prime moduli (no collision guarantee, probabilistic) |
| `MultiHeadEmbedding` | Read hash addresses | Single table + per-head offsets |
| Gate | Route embedding into residual | sqrt-sigmoid of dot-product |
| `ShortConv` | Local aggregation | kernel=4, dilation=3, depthwise, causal |
| `NanoEngram` | Full layer | hash → embed → gate → ShortConv → residual |
| mHC (skipped) | Diversity of key projections | Requires Riemannian optimizer — see sidebar |

**What's next:** Tutorial 21 trains all four model variants (NanoGPT, NanoDeepSeek, NanoDeepSeek+small Engram, NanoDeepSeek+medium Engram) and performs controlled ablations.

---

## Exercises

**1.** The hash mapping is probabilistic — two distinct n-grams may collide to the same table address. Estimate the theoretical collision rate for bigrams with `n` = 50,000 table size and GPT-2's 50,257-token vocabulary. Then measure empirically by running 10,000 bigrams through `NgramHashMapping` and counting collisions.

**2.** Replace the `_normalize` function with a stricter tokenizer that also strips punctuation. Does this reduce the compressed vocabulary further? Does it hurt downstream gating signal quality (measured by gate entropy)?

**3.** Remove the `ShortConv` and measure perplexity change (on a trained checkpoint from Tutorial 21). Is local aggregation important, or does the gating alone capture most of the benefit?

**4.** The gate uses `sqrt-sigmoid` rather than plain sigmoid. Plot both functions in the range $[-3, 3]$. How do they differ in the near-zero region? When does this matter for the Engram signal?

**5.** In the mHC discussion box, it says the orthogonality constraint requires Riemannian SGD. Implement a simple Gram-Schmidt orthogonalization step applied to the stack of $W_K^{(m)}$ projections after each optimizer step. Does training stability improve, even if true Riemannian gradients are not used?